#### 汇总md正文内容为一个连续的md文件
1. 找到所有符合模式的md文件，加入md list
2. 遍历md list，处理每一个md
    1. 遍历md每一行，过滤掉非正文信息（在每个md文件的前几行）以及页数分割符，例如：
    # 批次 16: 第 280-304 页
    处理时间: 2026-02-03 19:12:37
    总字符数: 35229
    ---
    {281}------------------------------------------------
    2. 过滤掉图名、表名和图片路径，图表名均为独立一行，以图|附图|附表|表开头，例如：
    图 1-1 浙江宁波保国寺大殿拼柱示意图
    ![图 1-1 浙江宁波保国寺大殿拼柱示意图](./images_all/page_19_Picture_6.png "图 1-1 浙江宁波保国寺大殿拼柱示意图")

    图 1-2 杭州灵隐寺石塔阑额“七朱八白”(五代末)
    ![图 1-2 杭州灵隐寺石塔阑额“七朱八白”(五代末)](./images_all/page_19_Picture_7.png "图 1-2 杭州灵隐寺石塔阑额“七朱八白”(五代末)")

    附表 2 功限比较表
    ![附表 2 功限比较表](./images_all/table_images/-2.png "附表 2 功限比较表")

    3. 保留标题和正文内容，按照顺序加入到一个新的汇总md。标题均以md语法 # XXXX起，且标题为独立一行

3. 汇总的仅包含标题和正文内容的md文件进行格式调整，标题和正文直接空一行，被页码分割符分割的语义不完整的段落，自动拼合，例如：
    《法式》小木作制度中提到许多间广尺寸,《研究》认为它们都以六等材为准,但这是不正

    {285}------------------------------------------------

    确的。如卷七及卷二十一小木作制度与功限《阑槛钩窗》里分别有:"槛面高一尺八寸至二尺。""阑槛一间高一尺八寸,广一丈二尺。
    需要拼接为：    《法式》小木作制度中提到许多间广尺寸,《研究》认为它们都以六等材为准,但这是不正确的。如卷七及卷二十一小木作制度与功限《阑槛钩窗》里分别有:"槛面高一尺八寸至二尺。""阑槛一间高一尺八寸,广一丈二尺。
4. 对汇总文件进行统一半角、圆角符号等格式化操作，用于后续rag chunks切分

In [7]:
import re
import os
from pathlib import Path

class MarkdownMainTextGatherer:
    def __init__(self, md_root_dir, output_file):
        self.md_root_dir = Path(md_root_dir)
        self.output_file = Path(output_file)
        # 匹配模式：图/附图/附表/表 开头的行
        self.fig_table_pattern = re.compile(r'^(图|附图|附表|表)\s?\d+.*')
        # 匹配图片语法：![alt](path)
        self.image_link_pattern = re.compile(r'^!\[.*\]\(.*\)')
        # 匹配页码分隔符：{281}-----------------------
        self.page_divider_pattern = re.compile(r'^\{\d+\}-+')
        # 过滤元数据的关键字
        self.metadata_keywords = ["批次", "处理时间", "总字符数", "---"]

    def find_md_files(self) -> list:
        """查找符合命名模式的md文件并排序"""
        md_files = []
        pattern = re.compile(r'batch\d+_page\d+-\d+_clean\.md$')
        for file_path in self.md_root_dir.rglob('*.md'):
            if pattern.search(file_path.name):
                md_files.append(file_path)
        # 这里的排序很重要，确保文件按顺序拼合
        return sorted(md_files)

    def normalize_text(self, text: str) -> str:
        """格式化操作：统一全角半角符号（根据RAG常规需求，通常保留中文字符习惯）"""
        # 示例：将连续的多个换行缩减为一个（在合并阶段会处理，这里做标点清洗）
        # 如果需要将英文标点转全角或反之，可在此添加映射
        # 这里演示最常见的：去除行首尾多余空格
        return text.strip()

    def is_valid_line(self, line: str) -> bool:
        """判断是否为有效的正文或标题行"""
        line = line.strip()
        if not line:
            return False
        # 过滤元数据行
        if any(keyword in line for keyword in self.metadata_keywords):
            return False
        # 过滤页码分割线
        if self.page_divider_pattern.match(line):
            return False
        # 过滤图表标题
        if self.fig_table_pattern.match(line):
            return False
        # 过滤图片路径
        if self.image_link_pattern.match(line):
            return False
        return True

    def process(self):
        md_files = self.find_md_files()
        print(f"找到 {len(md_files)} 个匹配的 Markdown 文件。")

        all_content_blocks = []

        for file_path in md_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    clean_line = line.strip()
                    if self.is_valid_line(clean_line):
                        all_content_blocks.append(clean_line)

        # 执行语义拼合与格式化逻辑
        final_lines = []
        if not all_content_blocks:
            print("没有找到有效内容。")
            return

        for i in range(len(all_content_blocks)):
            current_line = all_content_blocks[i]
            
            # 1. 如果是标题，确保前后格式
            if current_line.startswith('#'):
                # 如果上一行不是空的，可以在逻辑最后处理
                # final_lines.append("\n" + current_line + "\n")
                final_lines.append(current_line)
                continue

            # 2. 语义拼接逻辑：判断当前行是否结束
            # 如果当前行不是以句号、感叹号、问号、引号结尾，且下一行不是标题
            # 则认为下一行是本行的延续
            is_sentence_end = current_line.endswith(('。', '！', '？', '”', '；', '.', '!', '?', '"'))
            
            if not is_sentence_end and (i + 1 < len(all_content_blocks)):
                next_line = all_content_blocks[i+1]
                if not next_line.startswith('#'):
                    # 拼接到当前行，不加换行符
                    all_content_blocks[i+1] = current_line + next_line
                    continue
            
            # 如果是正常结束，或者是最后一行，或者是下一行是标题，则作为独立段落输出
            final_lines.append(current_line)

        # 写入文件
        with open(self.output_file, 'w', encoding='utf-8') as f:
            full_text = "\n\n".join([l for l in final_lines if l.strip()])
            # 统一简单的全角处理（如需要）
            full_text = full_text.replace('?', '?').replace('!', '!') 
            f.write(full_text)
            
        print(f"汇总完成！文件已保存至: {self.output_file}")

# --- Jupyter Cell 运行示例 ---
# 设置你的路径
md_root = r"knowledgeBase\pdfParse\cleaned_data"  # 替换为你的 md 所在文件夹
output_md = r"knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus.md"

gatherer = MarkdownMainTextGatherer(md_root, output_md)
gatherer.process()

找到 17 个匹配的 Markdown 文件。
汇总完成！文件已保存至: knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus.md


#### 为章节标题增加路径栈

In [8]:
import json

# --- 参数设置 ---
input_file = r'knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc.jsonl'  # 你的原始文件路径
output_file = r'knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc2.jsonl'  # 输出文件路径

# --- 核心处理逻辑 ---
def process_catalog(in_path, out_path):
    # 用于存储当前的路径层级，Key 是层级(int)，Value 是标题(str)
    # 例如: {1: "第一章", 2: "一、..."}
    current_path_map = {}
    
    processed_data = []

    try:
        with open(in_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                
                # 1. 解析 JSON
                item = json.loads(line)
                level = item.get("层级")
                title = item.get("章节标题")
                
                # 2. 更新当前层级路径
                # 将当前层级及更深的层级全部清理掉，确保路径正确回溯
                current_path_map[level] = title
                
                # 3. 构建 标题总路径 列表
                # 选取出从 1 到当前 level 的所有标题
                full_path = [current_path_map[i] for i in range(1, level + 1) if i in current_path_map]
                
                # 4. 插入新字段
                item["标题总路径"] = full_path
                processed_data.append(item)

        # 5. 写入新文件
        with open(out_path, 'w', encoding='utf-8') as f:
            for entry in processed_data:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
                
        print(f"处理完成！文件已保存至: {out_path}")
        
    except Exception as e:
        print(f"处理过程中出错: {e}")

# 执行函数
process_catalog(input_file, output_file)

处理完成！文件已保存至: knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc2.jsonl


#### 输出images的元数据，包括图索引、图名、路径、所在页码等等

In [9]:
import re
import json
from pathlib import Path
from collections import defaultdict

# ==================== 参数配置区 ====================
# 建议在 Jupyter 中直接修改这些字符串路径
config = {
    "md_root_dir": r"knowledgeBase\pdfParse\cleaned_data",
    "ref_table_path": r"knowledgeBase\pdfParse\cleaned_data\imagesName_Cleaned.md",
    "jsonl_output_path": r"knowledgeBase\pdfParse\cleaned_data\images_metadata.jsonl",
    "update_md_files": False # 是否同步修改原 Markdown 文件内容
}

# ==================== 核心逻辑区 ====================
class MarkdownImageRefiner:
    def __init__(self, cfg):
        self.cfg = {k: (Path(v) if k != "update_md_files" else v) for k, v in cfg.items()}
        self.reference_map = {}  # {归一化索引: {"index": 原索引, "name": 原名称, "page": 页码}}
        self.global_occurrences = defaultdict(list)
        
        self._load_reference_table()

    def _normalize(self, text):
        if not text: return ""
        return re.sub(r'\s+', '', text).replace('：', ':').replace('-', '-')

    def _int_to_chinese(self, n):
        """1-99 整数转中文数字"""
        chars = "零一二三四五六七八九"
        if n < 10: return chars[n]
        if n < 20: return "十" + (chars[n % 10] if n % 10 != 0 else "")
        unit = n % 10
        return chars[n // 10] + "十" + (chars[unit] if unit != 0 else "")

    def _load_reference_table(self):
        """解析参考表，提取 索引、名称、页码"""
        if not self.cfg['ref_table_path'].exists():
            print(f"❌ 找不到参考文件: {self.cfg['ref_table_path']}")
            return

        with open(self.cfg['ref_table_path'], 'r', encoding='utf-8') as f:
            for line in f:
                if '|' not in line or '---' in line or '索引' in line: continue
                # 假设格式: | 索引 | 名称 | 页码 |
                parts = [p.strip() for p in line.split('|') if p.strip()]
                if len(parts) >= 3:
                    idx_part, name_part, page_part = parts[0], parts[1], parts[2]
                    norm_idx = self._normalize(idx_part)
                    self.reference_map[norm_idx] = {
                        "index": idx_part,
                        "name": name_part,
                        "page": page_part
                    }
        print(f"✅ 成功加载参考表，共计 {len(self.reference_map)} 条图名记录。")

    def _extract_index(self, line):
        match = re.search(r'(图|附图|附表|表)\s*([\dA-Za-z\-\.]+)', line)
        return self._normalize(match.group(0)) if match else ""

    def process(self):
        # 1. 扫描所有文件
        md_files = sorted([f for f in self.cfg['md_root_dir'].rglob('*.md') 
                          if re.search(r'batch\d+_page\d+-\d+_clean\.md$', f.name)])
        
        for md_file in md_files:
            with open(md_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            # 记录该文件中图片和标题的行索引
            img_indices = [i for i, l in enumerate(lines) if re.search(r'!\[.*?\]\((.*?)\)', l)]
            
            for img_idx in img_indices:
                # 在图片行前后2行寻找标题
                search_range = range(max(0, img_idx-2), min(len(lines), img_idx+3))
                for line_idx in search_range:
                    norm_idx = self._extract_index(lines[line_idx])
                    if norm_idx in self.reference_map:
                        img_path_match = re.search(r'!\[.*?\]\((.*?)\)', lines[img_idx])
                        raw_path = img_path_match.group(1).split()[0] if img_path_match else ""
                        
                        # 记录匹配详情
                        self.global_occurrences[norm_idx].append({
                            "file_path": md_file,
                            "img_line_idx": img_idx,
                            "cap_line_idx": line_idx,
                            "raw_img_path": raw_path
                        })
                        break

        # 2. 生成结果 (处理其一、其二逻辑)
        jsonl_data = []
        file_updates = defaultdict(dict) # {file_path: {line_idx: new_content}}

        for norm_idx, occs in self.global_occurrences.items():
            ref = self.reference_map[norm_idx]
            total = len(occs)
            
            for i, occ in enumerate(occs):
                # 确定最终显示名称
                suffix = f"其{self._int_to_chinese(i+1)}" if total > 1 else ""
                final_name = f"{ref['name']}{suffix}"
                
                # 构造 JSONL 数据行
                abs_path = (occ['file_path'].parent / occ['raw_img_path']).resolve()
                jsonl_data.append({
                    "图/表索引": ref['index'],
                    "图/表名称": final_name,
                    "所在页码": int(ref['page']) if str(ref['page']).isdigit() else ref['page'],
                    "本地绝对路径": str(abs_path)
                })
                
                # 如果需要更新原文件内容
                if self.cfg['update_md_files']:
                    standard_block = f"{final_name}\n![{final_name}]({occ['raw_img_path']} \"{final_name}\")\n"
                    file_updates[occ['file_path']][occ['cap_line_idx']] = standard_block
                    file_updates[occ['file_path']][occ['img_line_idx']] = ""

        # 3. 执行写入
        self._write_outputs(jsonl_data, file_updates)

    def _write_outputs(self, jsonl_data, file_updates):
        # 写入 JSONL
        self.cfg['jsonl_output_path'].parent.mkdir(parents=True, exist_ok=True)
        with open(self.cfg['jsonl_output_path'], 'w', encoding='utf-8') as f:
            for item in jsonl_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        print(f"🚀 JSONL 文件已导出: {self.cfg['jsonl_output_path']} (共 {len(jsonl_data)} 条)")

        # 写入原文件修改
        if self.cfg['update_md_files']:
            for f_path, mods in file_updates.items():
                with open(f_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                for ln_idx, content in mods.items():
                    lines[ln_idx] = content
                
                new_content = re.sub(r'\n{3,}', '\n\n', "".join(lines))
                with open(f_path, 'w', encoding='utf-8') as f:
                    f.write(new_content)
            print(f"✨ 原 Markdown 文件已完成标准化重命名。")

# ==================== 执行单元 ====================
refiner = MarkdownImageRefiner(config)
refiner.process()

✅ 成功加载参考表，共计 386 条图名记录。
🚀 JSONL 文件已导出: knowledgeBase\pdfParse\cleaned_data\images_metadata.jsonl (共 428 条)


#### 输出注释索引字典，包括字段，章节、编号、注释内容

In [6]:
import re
import json

# --- 参数设置 ---
input_file = r'knowledgeBase\pdfParse\cleaned_data\allChapters_annotation.md'  # 你的输入文件名
output_file = r'knowledgeBase\pdfParse\cleaned_data\allChapters_annotation.jsonl' # 输出文件名

# --- 工具函数：中文数字转阿拉伯数字 (简单实现) ---
def cn_to_int(cn_str):
    cn_map = {'一': 1, '二': 2, '三': 3, '四': 4, '五': 5, '六': 6, '七': 7, '八': 8, '九': 9, '十': 10}
    if len(cn_str) == 1: return cn_map.get(cn_str, 0)
    if len(cn_str) == 2 and cn_str[0] == '十': return 10 + cn_map.get(cn_str[1], 0)
    if len(cn_str) == 2 and cn_str[1] == '十': return cn_map.get(cn_str[0], 0) * 10
    if len(cn_str) == 3: return cn_map.get(cn_str[0], 0) * 10 + cn_map.get(cn_str[2], 0)
    return 0

# --- 核心逻辑 ---
results = []
stats = {} # 用于存储过程数据
current_chapter_num = 0

# 正则表达式说明：
# 章节：匹配 ## 第(某)章
re_chapter = re.compile(r'^##\s*第([一二三四五六七八九十百]+)章')
# 注释：匹配 - (数字) 或 -(数字) 后的所有内容
re_note = re.compile(r'^\s*-\s*\((\d+)\)\s*(.*)')

with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        
        # 1. 匹配章节
        chapter_match = re_chapter.match(line)
        if chapter_match:
            cn_num = chapter_match.group(1)
            current_chapter_num = cn_to_int(cn_num)
            stats[f"第{cn_num}章"] = 0
            continue
            
        # 2. 匹配注释内容
        note_match = re_note.match(line)
        if note_match and current_chapter_num > 0:
            note_id = int(note_match.group(1))
            content = note_match.group(2).strip()
            
            item = {
                "chapter": current_chapter_num,
                "index": note_id,
                "text": content
            }
            results.append(item)
            
            # 更新统计数据
            chapter_key = list(stats.keys())[-1]
            stats[chapter_key] += 1

# --- 写入 JSONL ---
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in results:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

# --- 输出过程报告 ---
print("### 处理过程报告 ###")
for ch, count in stats.items():
    print(f"- {ch}: 找到 {count} 条注释")
print("-" * 20)
print(f"总计：生成 {len(results)} 条 JSONL 数据。")

### 处理过程报告 ###
- 第一章: 找到 35 条注释
- 第二章: 找到 31 条注释
- 第三章: 找到 2 条注释
- 第四章: 找到 9 条注释
- 第六章: 找到 5 条注释
- 第八章: 找到 4 条注释
- 第九章: 找到 2 条注释
--------------------
总计：生成 88 条 JSONL 数据。


#### chunks分块与元数据解析类
1. 给定一个仅包含带有层级的章节标题和正文的md文件，经过chunks分块与元数据解析器，输出包含丰富元数据的chunks jsonl文件，并输出中间过程report.md。给定md文件示例、输出jsonl示例如下：
......
# 第三章  铺作

铺作是木构架(大木作)的一部分,因其结构复杂与地位特殊而单立一章。

## 一、概说

当人们走近佛光寺大殿这座唐代殿堂的阶前,走进室内时,其疏朗雄大的斗栱给人以强烈的感受。中国古代建筑远看屋顶、近看斗栱这两个最醒目的特色,在这里得到淋漓尽致的表现。到了编写《营造法式》的年代,斗栱的装饰价值已逐渐被夸张,那些琳琅满目的铺作,有不少部分已失去了原来的结构价值而有了独立的装饰意义:一些房屋开间并不大,也排列了补间铺作;室内纵横罗列的大量斗栱,也是为了承载天花以及烘托皇权和神灵的至高无上。出现这种倾向,当然和北宋时期整个官式建筑追求精巧华丽的总趋势是分不开的,而这种风气沿袭至明清,导致斗栱的累赘程度达到了无以复加的地步。从《法式》所录斗栱图样来看,编者也着眼于复杂的、装饰性强的重栱全计心铺作,对一些简单斗栱和偷心造、单栱造则比较忽视,但恰好是这些简单的做法还较多地保留着斗栱原本的价值和意义。为了使这个曾在我国建筑历史上绽放过异彩的创造不致被追求繁褥豪华的风气所掩盖,我们应该努力让那些简朴真实的铺作恢复其应有的地位。

### （一）出跳承檐

铺作的基本功能是承托悬出的屋檐,其他的承梁、承天花、承平座等功能都是由此衍生的(图 3-1~3-4)。

曾闻老工匠有句口诀叫做"檐不过步",意为出檐不能超过步架,否则就有倾覆的危险, 而在保证出檐安全方面,斗栱起着关键作用。对此,可以用《法式》的相关规定,来对这个原理 进行一番检验:(1)以一座三间小厅堂为例,按《法式》规定可用六等材。如架深用6尺,椽径用0.3尺,则 檐出为3.5尺,飞子出跳为2.1尺,总檐出为5.6尺。如用柱梁作或单斗只替(无斗栱出跳),则 总檐出与架深比为5.6:6等于1:1.07,虽在"檐不过步"范围之内,但比例接近1:1,安全 系数差,如遇大风、大雪、上屋维修甚至地震等突发事件,屋檐垮塌的可能性极大。
......

{”chunk_content“:"当人们走近佛光寺大殿这座唐代殿堂的阶前，走进室内时，其疏朗雄大的斗供给人以强烈的感受......",
"metadata":{
    "chunk_id":"yingzao_fashi_chap3_001", # 务级、可解释 ID
    "chunk_size":389, 
    
    # 一、书籍信息
    "book_info": "《营造法式》解读(修订版)潘谷西 何建中 著",

    # 二、所在章节信息
    "closest_title":"1. 朵数"
    "toc_path": [ "第三章 铺作","一、概述", "（三）补间铺作的布置","1. 朵数"],
    
    #三、多模态
    "has_image":True,
    "images": [
    {
    "figure_id": "图1-1",
    "caption": "浙江宁波保国寺大殿拼柱示意图",
    "path": "本地绝对路径"
    }]，
    #四、正文注解
    "has_annotation":True,
    "annotation":[
    {
    "annotation_id":"chapter_1_index_1"
    "annotation_text":"- (1)《辽宁牛梁河红山文化女神庙与积冢群发掘简报》, 1986年8月《文物》"
    }，
    {
    "annotation_id":"chapter_1_index_2"
    "annotation_text":"- (2) 日僧圆仁:《入唐求法巡礼行记》卷二、卷三。"
    }
    ]
    }
}
2. 包含以下功能：chunks分块器和当前chunks元数据提取器
    1. 子类1：chunks分块类：滑动窗口选择一定行数的正文文本，基于段落分块，不可在同一段落的句子中进行切分，不可出现夸标题的chunk块。切分后的chunksize最大值支持参数设置，默认值为500中文字符。无overlap
    2. 子类2:元数据解析类：对当前切分的chunks进行元数据解析：下面逐字段介绍提取逻辑逻辑
        1. chunk_id，给出自定义前缀参数+章节信息+自动编号，唯一值。记录chunk数量，写入report.md
        1.2 chunk_size：计算字符串长度并写入
        2. book_info：根据给出自定义字符串参数，所有chunks统一写入
        3. closest_title：向上查找，距离当前文本最近的标题项，然后通过匹配标题名称，从外部章节信息jsonl文件的“标题总路径”字段中获取toc_path。记录外部jsonl文件每一行被匹配的次数，写入到report.md中章节索引部分
        4. 子类3:图片解析类：
            1. has_image：编写正则规则，匹配特定模式(图 1-8~1-10)、如图 2-30、(图 9-1、9-2)、（图2-1）、如图三十二、如图所示(图二十八至三十一)、(参见图 2-43)、(附图 2)、(附图4~8)、(表 2-2)、(附表 1)，匹配成功则为true，否则为false
            2. images：如当前文本匹配正则规则，则提取并解析所有的图索引为：图 1-1、图 9-10、附表 3、附图 10格式，加入一个list，注意(图 1-8~1-10)需要解析为1-8、1-9、1-10三张图
            3. 根据图索引list，到给定的所有images数据jsonl文件中，通过匹配“图/表索引”字段，获取“图/表名称”、“本地绝对路径”字段，然后分别填入元数据的images list中。记录外部jsonl文件每一行被匹配的次数，写入到report.md中图片索引部分
        5. 子类4:注解解析类：
            1. has_annotation：编写正则规则，匹配特定模式（<sup>1</sup>），如匹配成功，为true
            2. annotation：通过toc_path的第一个元素，获取到章节信息，然后解析当前（<sup>1</sup>）中的数字作为编号，组成chapter_n_index_m格式作为annotation_id；annotation_text则通过给定的外部引用jsonl文件查找，然后提取写入。记录外部jsonl文件每一行被匹配的次数，写入到report.md中注释索引部分
        6. 下面给出元数据解析器用到的外部jsonl文件示例：
            1. 章节信息jsonl：
            {"章节标题": "第一章  总论", "层级": 1,  "页码": 16, "标题总路径": ["第一章  总论"]}
            {"章节标题": "一、《营造法式》的性质与特点", "层级": 2, "页码": 16, "标题总路径": ["第一章  总论", "一、《营造法式》的性质与特点"]}
            {"章节标题": "（一）“营造法式”是一种建筑工程预算定额", "层级": 3, "标题总路径": ["第一章  总论", "一、《营造法式》的性质与特点", "（一）“营造法式”是一种建筑工程预算定额"], "页码": 16}
            {"章节标题": "（二）李诫《营造法式》的编写体例", "层级": 3, "页码": 17, "标题总路径": ["第一章  总论", "一、《营造法式》的性质与特点", "（二）李诫《营造法式》的编写体例"]}
            2. 注释信息jsonl：
            {"chapter": 1, "index": 1, "text": "《宋会要辑稿》第七十五册,职官三〇。"}
            {"chapter": 1, "index": 2, "text": "《宋会要辑稿》第七十五册,职官三〇。"}
            3. images信息jsonl：
            {"图/表索引": "图 1-1", "图/表名称": "浙江宁波保国寺大殿拼柱示意图", "所在页码": 19, "本地绝对路径": "D:\\postgraduate_study\\graduation_thesis\\llm_rag\\myArAppRag\\knowledgeBase\\pdfParse\\cleaned_data\\images_all\\page_19_Picture_6.png"}
            {"图/表索引": "附表 1", "图/表名称": "南宋永思陵建筑尺度表", "所在页码": 282, "本地绝对路径": "D:\\postgraduate_study\\graduation_thesis\\llm_rag\\myArAppRag\\knowledgeBase\\pdfParse\\cleaned_data\\images_all\\table_images\\-1.png"}


优化逻辑：
1. 图片解析方法：增加“和”、“及”、“与”、“到”等连接词优化
2. 部分正文内容包含(图 2-13-A~2-13-E)，对应的外部jsonl图索引为：图 2-13-A，需要兼容这种情况
3. 注释、图片、章节，与外部jsonl匹配时，去除多余空格、统一符号后，采用计算字符串相似度，相似度>70，即可认为匹配成功
4. 同一chunk内解析到的图片、注解，要去重复，避免因为识别误差添加多次一样的元数据，图片可根据本地绝对路径验重，注释则根据annotation_id
5. 如注释、图片、章节，与外部jsonl匹配失败，report中记录该chunk文本所在md文件的行数，便于人工复核

In [1]:
import json
import re
import os
import unicodedata

# ==========================================
# 1. 通用工具与算法函数
# ==========================================
def cn2arabic(cn_str):
    """中文数字转阿拉伯数字"""
    if str(cn_str).isdigit(): return int(cn_str)
    cn_num_dict = {'零':0, '一':1, '二':2, '三':3, '四':4, '五':5, '六':6, '七':7, '八':8, '九':9, '十':10, '百':100, '千':1000}
    unit_dict = {'十': 10, '百': 100, '千': 1000}
    if cn_str.startswith('十') and len(cn_str) > 1: cn_str = '一' + cn_str
    result, temp = 0, 0
    for char in cn_str:
        if char in unit_dict:
            unit = unit_dict[char]
            if temp == 0: temp = 1
            result += temp * unit
            temp = 0
        elif char in cn_num_dict:
            temp = cn_num_dict[char]
    result += temp
    return result

def normalize_text(text):
    """文本标准化：全半角统一、中英文符号统一、去空格、转小写"""
    if not text: return ""
    text = str(text)
    
    # 1. NFKC 标准化：自动处理全角转半角 (如 １２３ -> 123, ＡＢＣ -> ABC)
    text = unicodedata.normalize('NFKC', text)
    # 2. 转小写
    text = text.lower()
    # 3. 统一中英文标点符号 (将常见的中文标点映射为对应的英文标点)
    trans_table = str.maketrans("，。！？；：‘’“”（）【】", ",.!?;:''\"\"()[]")
    text = text.translate(trans_table)
    # 4. 去除所有空白字符（空格、换行、制表符等）
    text = re.sub(r'\s+', '', text)
    
    return text

def exact_match(query, candidates):
    """基于标准化后的严格字符串比较"""
    if not query: return None
    query_norm = normalize_text(query)
    
    for cand in candidates:
        if query_norm == normalize_text(cand):
            return cand
            
    return None

def expand_image_indices(img_str):
    """展开图号支持连字符和字母"""
    match = re.match(r'([图表]|附[图表])\s*(.+)', img_str.strip())
    if not match: return []
    prefix, content = match.groups()
    
    content = re.sub(r'[和及与]', '、', content)
    content = re.sub(r'[至到]', '~', content)
    
    if '、' in content:
        return [f"{prefix} {p.strip()}" for p in content.split('、')]
    
    if '~' in content:
        start_str, end_str = [s.strip() for s in content.split('~', 1)]
        if re.search(r'[A-Za-z]$', start_str) and re.search(r'[A-Za-z]$', end_str):
            base = start_str[:-1]
            start_char, end_char = start_str[-1], end_str[-1]
            return [f"{prefix} {base}{chr(i)}" for i in range(ord(start_char), ord(end_char) + 1)]
            
        prefix_main = ""
        if '-' in start_str:
            main_parts = start_str.rsplit('-', 1)
            prefix_main = main_parts[0] + '-'
            start_num_str = main_parts[1]
            end_num_str = end_str.rsplit('-', 1)[-1] if '-' in end_str else end_str
        else:
            start_num_str, end_num_str = start_str, end_str
            
        start_idx, end_idx = cn2arabic(start_num_str), cn2arabic(end_num_str)
        return [f"{prefix} {prefix_main}{i}" for i in range(start_idx, end_idx + 1)]

    num = cn2arabic(content) if not '-' in content and not re.search(r'[A-Za-z]', content) else content
    return [f"{prefix} {num}"]


# ==========================================
# 2. 核心处理类
# ==========================================
class MarkdownProcessor:
    def __init__(self, book_info, prefix_id, max_chunk_size=500):
        self.book_info = book_info
        self.prefix_id = prefix_id
        self.max_chunk_size = max_chunk_size
        
        self.db_chapters = {}    # key: 原始标题
        self.db_annotations = {} # key: (chapter_num, index_num)
        self.db_images = {}      # key: 原始图表索引
        
        self.report_stats = {
            "chunks_total": 0,
            "chapter_hits": {}, "chapter_misses": [],
            "image_hits": {}, "image_misses": [],
            "annotation_hits": {}, "annotation_misses": []
        }

    def load_external_dbs(self, chapters_path, annotations_path, images_path):
        if os.path.exists(chapters_path):
            with open(chapters_path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    title = data["章节标题"].strip()
                    self.db_chapters[title] = data
                    self.report_stats["chapter_hits"][title] = 0
                    
        if os.path.exists(annotations_path):
            with open(annotations_path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    anno_key = (data["chapter"], data["index"])
                    self.db_annotations[anno_key] = data
                    self.report_stats["annotation_hits"][anno_key] = 0
                    
        if os.path.exists(images_path):
            with open(images_path, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    idx = data["图/表索引"].strip()
                    self.db_images[idx] = data
                    self.report_stats["image_hits"][idx] = 0
    def _extract_images_from_text(self, text, start_line):
        # 匹配模式保持不变
        pattern = r'[（(]?\s*(?:如图|参见)?(?:所示)?\s*([图表]|附[图表])\s*([A-Za-z一二三四五六七八九十百0-9\-\~\s至及和与到、]+)[）)]?'
        matches = re.finditer(pattern, text)
        
        extracted_images = []
        seen_paths = set()
        
        for match in matches:
            full_raw_text = match.group(0)  # 整个匹配到的文本，如 "(如图 1-1)"
            raw_img_label = match.group(1) + " " + match.group(2) # 拼接出的搜索基准
            
            # 展开图号（如 "图 1~3" 展开为 ["图 1", "图 2", "图 3"]）
            expanded_indices = expand_image_indices(raw_img_label)
            
            for img_idx in expanded_indices:
                matched_key = exact_match(img_idx, self.db_images.keys())
                
                if matched_key:
                    self.report_stats["image_hits"][matched_key] += 1
                    img_data = self.db_images[matched_key]
                    path = img_data.get("本地绝对路径", "")
                    
                    if path not in seen_paths:
                        seen_paths.add(path)
                        extracted_images.append({
                            "figure_id": matched_key,
                            "caption": img_data.get("图/表名称", ""),
                            "path": path
                        })
                else:
                    # 【核心修改】：在警告信息中包含原始捕获文本
                    msg = (f"起始行号 {start_line} | "
                           f"原始文本: 『{full_raw_text}』 | "
                           f"未匹配索引: [{img_idx}]")
                    self.report_stats["image_misses"].append(msg)
                    
        return extracted_images

    def _extract_annotations_from_text(self, text, toc_path, start_line):
        extracted_annotations = []
        seen_ids = set()
        pattern = r'<sup>[（(](\d+)[）)]</sup>'
        matches = re.finditer(pattern, text)
        
        chapter_num = -1
        if toc_path:
            cap_match = re.search(r'第([一二三四五六七八九十百0-9]+)章', toc_path[0])
            if cap_match: chapter_num = cn2arabic(cap_match.group(1))

        for match in matches:
            idx_num = int(match.group(1))
            anno_key = (chapter_num, idx_num)
            anno_id = f"chapter_{chapter_num}_index_{idx_num}"
            
            if anno_id in seen_ids: continue
                
            if anno_key in self.db_annotations:
                self.report_stats["annotation_hits"][anno_key] += 1
                anno_data = self.db_annotations[anno_key]
                seen_ids.add(anno_id)
                extracted_annotations.append({
                    "annotation_id": anno_id,
                    "annotation_text": anno_data.get("text", "")
                })
            else:
                msg = f"起始行号 {start_line} | 未找到注释库映射: 章节 {chapter_num}, 索引 {idx_num}"
                self.report_stats["annotation_misses"].append(msg)
                
        return extracted_annotations

    def process_markdown(self, md_filepath, output_jsonl, report_md):
        with open(md_filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        chunks = []
        current_toc = []
        current_chunk_paragraphs = []
        current_chunk_size = 0
        chunk_start_line = 1
        
        def save_chunk(end_line_num):
            nonlocal current_chunk_paragraphs, current_chunk_size, chunk_start_line
            if not current_chunk_paragraphs: return
            
            chunk_text = "\n".join(current_chunk_paragraphs)
            closest_title = current_toc[-1].lstrip('#').strip() if current_toc else ""
            
            toc_path = []
            matched_title_key = exact_match(closest_title, self.db_chapters.keys()) if closest_title else None
            
            if matched_title_key:
                self.report_stats["chapter_hits"][matched_title_key] += 1
                toc_path = self.db_chapters[matched_title_key].get("标题总路径", [])
            else:
                msg = f"起始行号 {chunk_start_line} | 未找到章节库映射: {closest_title}"
                if closest_title: 
                    self.report_stats["chapter_misses"].append(msg)

            chap_num = "0"
            if toc_path:
                cap_match = re.search(r'第([一二三四五六七八九十百0-9]+)章', toc_path[0])
                if cap_match: chap_num = str(cn2arabic(cap_match.group(1)))

            self.report_stats["chunks_total"] += 1
            chunk_id = f"{self.prefix_id}_chap{chap_num}_{self.report_stats['chunks_total']:03d}"
            
            images = self._extract_images_from_text(chunk_text, chunk_start_line)
            annotations = self._extract_annotations_from_text(chunk_text, toc_path, chunk_start_line)

            chunks.append({
                "chunk_content": chunk_text,
                "metadata": {
                    "chunk_id": chunk_id,
                    "chunk_size": len(chunk_text),
                    "book_info": self.book_info,
                    "closest_title": closest_title,
                    "toc_path": toc_path,
                    "has_image": len(images) > 0,
                    "images": images,
                    "has_annotation": len(annotations) > 0,
                    "annotation": annotations
                }
            })
            
            current_chunk_paragraphs = []
            current_chunk_size = 0

        print("[*] 开始执行 Chunks 分块与元数据解析...")
        for i, line in enumerate(lines, 1):
            line_str = line.strip()
            if not line_str: continue
            
            if not current_chunk_paragraphs:
                chunk_start_line = i
                
            if line_str.startswith('#'):
                save_chunk(i)
                level = len(line_str) - len(line_str.lstrip('#'))
                current_toc = current_toc[:level-1]
                current_toc.append(line_str)
                chunk_start_line = i
                continue
            
            line_len = len(line_str)
            if current_chunk_size + line_len > self.max_chunk_size and current_chunk_size > 0:
                save_chunk(i)
                chunk_start_line = i
            
            current_chunk_paragraphs.append(line_str)
            current_chunk_size += line_len
            
        save_chunk(len(lines))

        print(f"[*] 写入 Chunk 结果到: {output_jsonl}")
        with open(output_jsonl, 'w', encoding='utf-8') as f:
            for c in chunks:
                f.write(json.dumps(c, ensure_ascii=False) + '\n')
                
        self._generate_report(report_md)
        print(f"[*] 处理完成！产出 Chunks 数量: {len(chunks)}。报告已保存至 {report_md}")

    def _generate_report(self, report_md):
        with open(report_md, 'w', encoding='utf-8') as f:
            f.write("# 📚 Chunk分块与元数据解析诊断报告\n\n")
            f.write(f"- 总生成 Chunk 数量: {self.report_stats['chunks_total']}\n\n")
            
            # --- 1. 命中统计 ---
            f.write("## 1. ✅ 外部数据源命中统计 (正文成功关联DB)\n")
            
            f.write("### 📖 章节命中\n")
            for k, v in self.report_stats["chapter_hits"].items():
                if v > 0: f.write(f"- `{k}`: 命中 {v} 次\n")
                
            f.write("\n### 🖼️ 图片命中 (去重后)\n")
            for k, v in self.report_stats["image_hits"].items():
                if v > 0: f.write(f"- `{k}`: 命中 {v} 次\n")
                
            f.write("\n### 📝 注释命中 (去重后)\n")
            for k, v in self.report_stats["annotation_hits"].items():
                if v > 0: f.write(f"- 第{k[0]}章 注释{k[1]}: 命中 {v} 次\n")
                
            # --- 2. 外部数据库未命中 ---
            f.write("\n---\n## 2. 📉 DB中存在但未被正文命中 (需要检查正文是否漏标)\n")
            
            unhit_chapters = [k for k, v in self.report_stats["chapter_hits"].items() if v == 0]
            if unhit_chapters:
                f.write("### 📖 未命中的章节\n")
                for k in unhit_chapters: f.write(f"- {k}\n")
            
            unhit_images = [k for k, v in self.report_stats["image_hits"].items() if v == 0]
            if unhit_images:
                f.write("\n### 🖼️ 未命中的图片\n")
                for k in unhit_images: f.write(f"- {k}\n")
                
            unhit_annos = [k for k, v in self.report_stats["annotation_hits"].items() if v == 0]
            if unhit_annos:
                f.write("\n### 📝 未命中的注释\n")
                for k in unhit_annos: f.write(f"- 第{k[0]}章 注释{k[1]}\n")
            
            if not (unhit_chapters or unhit_images or unhit_annos):
                f.write("🎉 完美！外部数据库提供的所有内容均在正文中被成功引用命中。\n")

            # --- 3. 缺失警告 ---
            f.write("\n---\n## 3. ⚠️ 正文提及但在DB中缺失 (需要检查外部数据库是否遗漏)\n")
            all_misses = (self.report_stats["chapter_misses"] + 
                          self.report_stats["image_misses"] + 
                          self.report_stats["annotation_misses"])
            if not all_misses:
                f.write("🎉 完美！正文中出现的所有元数据标识均在外部数据库中找到了来源。\n")
            else:
                for miss in sorted(set(all_misses)):
                    f.write(f"- {miss}\n")


In [4]:
# ==========================================
CONFIG = {
    # >>> 请按实际情况修改此处路径参数 <<<
    "md_input_path": r"knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus.md",
    "chapters_jsonl": r"knowledgeBase\pdfParse\cleaned_data\yingzaofashi_jiedu_v2_toc.jsonl",
    "annotations_jsonl": r"knowledgeBase\pdfParse\cleaned_data\allChapters_annotation.jsonl",
    "images_jsonl": r"knowledgeBase\pdfParse\cleaned_data\images_metadata.jsonl",
    
    "output_jsonl": r"knowledgeBase\chunks\yingzaofashi_jieduv2_chunks.jsonl",
    "report_md": r"knowledgeBase\pdfParse\dataAnalysis\chunks_process_report.md",
    
    "book_info_str": "《营造法式》解读(2017年3月修订版) 潘谷西 何建中著",
    "custom_prefix_id": "yingzao_fashi",
    "max_chunk_size": 500
}

# 确保输出目录存在
os.makedirs(os.path.dirname(CONFIG["output_jsonl"]), exist_ok=True)
os.makedirs(os.path.dirname(CONFIG["report_md"]), exist_ok=True)

# 实例化解析管线
processor = MarkdownProcessor(
    book_info=CONFIG["book_info_str"], 
    prefix_id=CONFIG["custom_prefix_id"], 
    max_chunk_size=CONFIG["max_chunk_size"]
)

# 1. 加载外部库建立匹配索引
processor.load_external_dbs(
    CONFIG["chapters_jsonl"], 
    CONFIG["annotations_jsonl"], 
    CONFIG["images_jsonl"]
)

# 2. 执行切分与抽取引擎
if os.path.exists(CONFIG["md_input_path"]):
    processor.process_markdown(
        CONFIG["md_input_path"], 
        CONFIG["output_jsonl"], 
        CONFIG["report_md"]
    )
else:
    print(f"[-] 提示：未找到输入文件 {CONFIG['md_input_path']}，请检查路径。")

[*] 开始执行 Chunks 分块与元数据解析...
[*] 写入 Chunk 结果到: knowledgeBase\chunks\yingzaofashi_jieduv2_chunks.jsonl
[*] 处理完成！产出 Chunks 数量: 501。报告已保存至 knowledgeBase\pdfParse\dataAnalysis\chunks_process_report.md
